# W90 vs QE comparison plotting

#### モジュール

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import ipynbname
NB_NAME = ipynbname.name()

#### データセットの定義

In [ ]:
# 公開用サンプル

dataset_material_a_wann96 = {
    "title": "MaterialA: QE vs Wannier90 comparison (example)",
    "qe": {
        "file": "./../materials/MaterialA/NC/SOC/bands/bands.out.gnu",
        "label": "QE",
        "shift": 0.0000,  # ダミー値
    },
    "w90": {
        "file": "./../materials/MaterialA/NC/SOC/wann96/MaterialA_band.dat",
        "label": "W90",
        "shift": 0.0000,  # ダミー値
    },
    "labelinfo": {
        "file": "./../materials/MaterialA/NC/SOC/wann96/MaterialA_band.labelinfo.dat",
    },
}

#### バンドファイル読み込み関数

In [ ]:
# 空行区切りのバンドファイル読み取り（QEの.gnu, W90の.datは同一フォーマットのため共用）
def read_separated_band_file(filename):
    bands_list = []
    current_band = []
    with open(filename) as f:
        for line in f:
            if line.strip() == "":
                if current_band:
                    bands_list.append(np.array(current_band))
                    current_band = []
            else:
                vals = line.split()
                current_band.append((float(vals[0]), float(vals[1])))
        if current_band:  # ファイル末尾に空行がない場合の対処
            bands_list.append(np.array(current_band))
    return bands_list

# バンドのリストをNaN区切りで1本の配列に結合する（plot用: 線が繋がらないようにする）
def concat_bands_with_nan(bands_list):
    pieces = []
    for band in bands_list:
        pieces.append(band)
        pieces.append(np.full((1, band.shape[1]), np.nan))
    return np.concatenate(pieces[:-1], axis=0)  # 末尾の余分なNaN行は除く

# 高対称点ラベル情報の読み取り（W90のlabelinfo.dat）
def read_labelinfo_file(filename):
    label_name = np.loadtxt(filename, dtype=str,  usecols=(0), unpack=True)
    label_loc  = np.loadtxt(filename, dtype=float, usecols=(2), unpack=True)
    return label_name, label_loc

#### プロット

In [ ]:
# データセットの指定
dataset = dataset_material_a_wann96
title = dataset["title"]

# サブプロットフレームの作成
fig, ax1 = plt.subplots(1, 1, figsize=(8, 6.0))
fig.canvas.header_visible = False

# --- W90バンドの描画 ---
w90_bands_list = read_separated_band_file(dataset["w90"]["file"])
w90_concat = concat_bands_with_nan(w90_bands_list)  # バンド間が繋がらないようにNaN区切りで1本化
ax1.plot(w90_concat[:, 0], w90_concat[:, 1] - dataset["w90"]["shift"],
          color='red', linewidth=1, linestyle='-', label=dataset["w90"]["label"])

# --- QEバンドの描画 ---
qe_bands_list = read_separated_band_file(dataset["qe"]["file"])
qe_concat = np.concatenate(qe_bands_list, axis=0)  # scatterは点ごとなので繋がる心配なし

# QE・W90のk軸スケーリング差を調整
w90_kmax = w90_bands_list[-1][-1, 0]
qe_kmax  = qe_bands_list[-1][-1, 0]
scaling_qe2wan = w90_kmax / qe_kmax

ax1.scatter(qe_concat[:, 0] * scaling_qe2wan, qe_concat[:, 1] - dataset["qe"]["shift"],
            color='black', s=3, label=dataset["qe"]["label"])

# --- 高対称点ラベルの設定 ---
label_name, label_loc = read_labelinfo_file(dataset["labelinfo"]["file"])
ax1.set_xlabel("K-point")
ax1.set_ylabel("Energy (eV)")
ax1.set_xticks(label_loc)
ax1.set_xticklabels(label_name)
ax1.set_xlim(0, w90_kmax)
ax1.legend(fontsize=10)
ax1.grid(True, linestyle=':')

fig.suptitle(title, fontsize=16)

# 画像として保存
save_dir = f"./{NB_NAME}_save"
os.makedirs(save_dir, exist_ok=True)
fig.savefig(f"{save_dir}/{re.sub(r'[^\w\-]+', '_', title)}.png", dpi=300, bbox_inches="tight")